<a href="https://colab.research.google.com/github/FatherNurt/FUNt-Cosmologiical-Model-of-All-Things/blob/main/06_FUNt_Validation_Suite_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FUNt Validation Suite v1.0
## Metrics, Confusion Matrix, and Event Review

Generated as part of the FUNt Transition Laboratory notebook suite.

## Purpose

Pure validation.

No physics is changed here. This notebook evaluates accumulated run logs:

- confusion matrix,
- MAE/RMSE,
- precision/recall/F1,
- false alarm review,
- missed storm review,
- suppressed transition review.

In [1]:
import json
import math
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

UTC_NOW = datetime.now(timezone.utc)
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

VERSION = "FUNt Validation Suite v1.0"
print(VERSION)


FUNt Validation Suite v1.0


In [2]:
def compute_binary_metrics(df, predicted_col="predicted_storm_binary", observed_col="observed_storm_binary"):
    valid = df[[predicted_col, observed_col]].dropna()
    if valid.empty:
        return {}
    pred = valid[predicted_col].astype(bool)
    obs = valid[observed_col].astype(bool)
    tp = int((pred & obs).sum())
    fp = int((pred & ~obs).sum())
    fn = int((~pred & obs).sum())
    tn = int((~pred & ~obs).sum())
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    f1 = 2*precision*recall/(precision+recall) if precision == precision and recall == recall and (precision+recall) else np.nan
    return {"tp":tp, "fp":fp, "fn":fn, "tn":tn, "precision":precision, "recall":recall, "f1":f1}

# Demo
demo = pd.DataFrame({
    "predicted_storm_binary": [False, False, True, True, False],
    "observed_storm_binary":  [False, True,  True, False, False],
})
print(compute_binary_metrics(demo))
display(pd.crosstab(demo["observed_storm_binary"], demo["predicted_storm_binary"],
                    rownames=["Observed"], colnames=["Predicted"]))


{'tp': 1, 'fp': 1, 'fn': 1, 'tn': 2, 'precision': 0.5, 'recall': 0.5, 'f1': 0.5}


Predicted,False,True
Observed,,
False,2,1
True,1,1
